In [1]:
import sqlite3


conn = sqlite3.connect("../podcasts.db")
cursor = conn.cursor()

conn.execute("PRAGMA foreign_keys = ON;")

try:
    conn.execute("BEGIN TRANSACTION")

    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS podcast (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            category TEXT NOT NULL,
            subcategory TEXT DEFAULT 'UNKNOWN',
            update_frequency TEXT DEFAULT 'UNKNOWN',
            rating INTEGER DEFAULT 0,
            reviews INTEGER DEFAULT 0,
            url TEXT NOT NULL,
            show_id TEXT NOT NULL UNIQUE,
            created_at TEXT NOT NULL,
            last_seen_at TEXT NOT NULL,
            updated_at TEXT DEFAULT CURRENT_TIMESTAMP
        );
        """
    )

    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS ranking (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            podcast_id INTEGER NOT NULL,
            country TEXT NOT NULL,
            rank INTEGER DEFAULT 0,
            chart_date TEXT NOT NULL,
            scraped_at TEXT NOT NULL,
            FOREIGN KEY(podcast_id) REFERENCES podcast(id)
        );
        """
    )

    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS score (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            podcast_id INTEGER NOT NULL,
            rank_score INTEGER DEFAULT 0,
            date TEXT NOT NULL,
            FOREIGN KEY(podcast_id) REFERENCES podcast(id)
        );
        """
    )

    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS download (
            id INTEGER PRIMARY KEY,
            status TEXT NOT NULL,
            scraped_at TEXT NOT NULL,
            FOREIGN KEY(id) REFERENCES podcast(id)
        );
        """
    )

    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS episode (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            podcast_id INTEGER NOT NULL,
            episode_id TEXT NOT NULL,
            show_title TEXT NOT NULL,
            show_id INTEGER NOT NULL,
            title TEXT,
            release_date TEXT,
            duration TEXT,
            has_free_version BOOL,
            channel_name TEXT,
            show_image TEXT,
            episode_image TEXT,
            show_alt_image TEXT,
            episode_url TEXT,
            summary TEXT,
            caption TEXT,
            short_caption TEXT,
            scraped_at TEXT NOT NULL,
            updated_at TEXT DEFAULT CURRENT_TIMESTAMP,
            UNIQUE(podcast_id, episode_id),
            FOREIGN KEY (podcast_id) REFERENCES podcast(id)
        );
        """
    )
    cursor.execute(
        """
        CREATE VIRTUAL TABLE IF NOT EXISTS episode_fts USING fts5(
            show_title,
            summary,
            content='episode',
            content_rowid='id'
        );
        """
    )
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS frequency (
            podcast_id INTEGER PRIMARY KEY,
            frequency TEXT NOT NULL,
            next_scrape TEXT NOT NULL,
            FOREIGN KEY (podcast_id) REFERENCES podcast(id)
            UNIQUE(podcast_id)
        );
        """
    )
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS meta (
            key TEXT PRIMARY KEY,
            value TEXT NOT NULL
        );
        """
    )
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS tag (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL UNIQUE
        );
        """
    )
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS episode_tag (
            episode_id INTEGER NOT NULL,
            tag_id INTEGER NOT NULL,
            PRIMARY KEY (episode_id, tag_id),
            FOREIGN KEY (episode_id) REFERENCES episode(id) ON DELETE CASCADE,
            FOREIGN KEY (tag_id) REFERENCES tag(id) ON DELETE CASCADE
        );
        """
    )

    cursor.execute("CREATE INDEX IF NOT EXISTS idx_show_id ON podcast (show_id);")
    cursor.execute(
        "CREATE INDEX IF NOT EXISTS idx_ranking_podcast_id ON ranking(podcast_id);"
    )
    cursor.execute(
        "CREATE INDEX IF NOT EXISTS idx_ranking_chart_date ON ranking(chart_date);"
    )
    cursor.execute(
        "CREATE INDEX IF NOT EXISTS idx_episode_podcast_id ON episode(podcast_id);"
    )
    cursor.execute(
        "CREATE INDEX IF NOT EXISTS idx_episode_release_date ON episode(release_date);"
    )
    cursor.execute(
        "CREATE INDEX IF NOT EXISTS idx_episode_tag_episode_id ON episode_tag(episode_id);"
    )
    cursor.execute(
        "CREATE INDEX IF NOT EXISTS idx_episode_tag_tag_id ON episode_tag(tag_id);"
    )

    cursor.execute(
        """
        CREATE TRIGGER IF NOT EXISTS trg_podcast_updated
        AFTER UPDATE ON podcast
        FOR EACH ROW
        BEGIN
            UPDATE podcast SET updated_at = CURRENT_TIMESTAMP WHERE id = OLD.id;
        END;
        """
    )
    cursor.execute(
        """
        CREATE TRIGGER IF NOT EXISTS trg_episode_updated
        AFTER UPDATE ON episode
        FOR EACH ROW
        BEGIN
            UPDATE episode SET updated_at = CURRENT_TIMESTAMP WHERE id = OLD.id;
        END;
        """
    )
    cursor.execute(
        """
        CREATE TRIGGER IF NOT EXISTS episode_ai AFTER INSERT ON episode BEGIN
            INSERT INTO episode_fts(rowid, show_title, summary)
            VALUES (new.id, new.show_title, new.summary);
        END;
        """
    )
    cursor.execute(
        """
        CREATE TRIGGER IF NOT EXISTS episode_ad AFTER DELETE ON episode BEGIN
            DELETE FROM episode_fts WHERE rowid = old.id;
        END;
        """
    )
    cursor.execute(
        """
        CREATE TRIGGER IF NOT EXISTS episode_au AFTER UPDATE ON episode BEGIN
            UPDATE episode_fts SET show_title = new.show_title, summary = new.summary
            WHERE rowid = old.id;
        END;
        """
    )

    conn.commit()

except sqlite3.Error as error:
    print(error)
    conn.rollback()

finally:
    conn.close()

In [2]:
from datetime import datetime
from pathlib import Path
import re
import sqlite3

from selectolax.parser import HTMLParser


def extract_list(html):
    tree = HTMLParser(html)
    main = tree.css_first("main")
    if not main:
        return ""
    ul = main.css_first("ul")
    return ul.html if ul else ""


def extract_link_info(a_tag):
    if a_tag:
        href = a_tag.attrs.get("href", "")
        text = a_tag.text(strip=True)
        return href, text
    return None


def extract_links_from_list(html):
    tree = HTMLParser(html)
    links = []

    for li in tree.css("li"):
        a_tag = li.css_first("a")
        if a_tag:
            link_info = extract_link_info(a_tag)
            if link_info:
                links.append(link_info)

    return links


def get_links(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        html = f.read()
    list_html = extract_list(html)
    return extract_links_from_list(list_html)


def extract_show_id(url):
    match = re.search(r"/id(\d+)", url)
    return match.group(1) if match else None


conn = sqlite3.connect("../podcasts.db")
cursor = conn.cursor()

timestamp = datetime.now().isoformat()
charts_dir = Path("../charts")
dirs = [
    dir for dir in charts_dir.iterdir() if ".DS_Store" not in str(dir) and dir.is_dir()
]

ranking_batch = []

conn.execute("BEGIN TRANSACTION")

for dir in dirs:
    country = dir.stem
    files = [file for file in dir.iterdir() if ".DS_Store" not in str(file)]
    for file in files:
        category = file.stem
        links = get_links(file)
        for i, (url, name) in enumerate(links):
            show_id = extract_show_id(url)
            if not show_id:
                continue

            cursor.execute(
                """
                INSERT OR IGNORE INTO podcast (
                    name, category, url, show_id, created_at, last_seen_at
                )
                VALUES (?, ?, ?, ?, ?, ?)
                """,
                (name, category, url, show_id, timestamp, timestamp),
            )

            cursor.execute(
                """
                UPDATE podcast SET last_seen_at = ? WHERE show_id = ?
                """,
                (timestamp, show_id),
            )

            cursor.execute(
                "SELECT id FROM podcast WHERE show_id = ?",
                (show_id,),
            )
            row = cursor.fetchone()
            if not row:
                continue
            podcast_id = row[0]

            ranking_batch.append((podcast_id, country, i + 1, timestamp, timestamp))

cursor.executemany(
    """
    INSERT INTO ranking (podcast_id, country, rank, chart_date, scraped_at)
    VALUES (?, ?, ?, ?, ?)
    """,
    ranking_batch,
)

conn.commit()
conn.close()

In [ ]:
import sqlite3
from datetime import datetime

date = datetime.now().isoformat()

conn = sqlite3.connect("../podcasts.db")
cursor = conn.cursor()

cursor.execute(
    """
    INSERT OR IGNORE INTO download (id, status, scraped_at)
    SELECT id, 'pending', ?
    FROM podcast
    """,
    (date,),
)

conn.commit()
conn.close()